In [1]:
import pandas as pd
import numpy as np

In [4]:
file_path = "Predict_data.xlsx"
Data_df = pd.read_excel(file_path, sheet_name=0,header=0)
Feature_df = pd.read_excel(file_path, sheet_name=1, header=0,index_col=0) 

In [5]:
Data_df

,d33,Tc,Pb-2,Ti-4,Bi-3,Sc-3,Ni-2,Zr-4,Mn-2,Zn-2,...,In-3,Nb-5,Sn-2,Cd-2,Sb-5,Mg-2,Li-1,Sb-3,Mn-3,W-6
0,0,0,0.64,0.6336,0.36,0.36,0.00032,0,0.00000,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000320
1,0,0,0.64,0.6272,0.36,0.36,0.00064,0,0.00000,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000640
2,0,0,0.64,0.6208,0.36,0.36,0.00096,0,0.00000,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000960
3,0,0,0.64,0.6144,0.36,0.36,0.00128,0,0.00000,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.001280
4,0,0,0.64,0.6336,0.36,0.36,0.00000,0,0.00032,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000320
5,0,0,0.64,0.6272,0.36,0.36,0.00000,0,0.00064,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000640
6,0,0,0.64,0.6208,0.36,0.36,0.00000,0,0.00096,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000960
7,0,0,0.64,0.6144,0.36,0.36,0.00000,0,0.00128,0.00000,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.001280
8,0,0,0.64,0.6336,0.36,0.36,0.00000,0,0.00000,0.00032,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000320
9,0,0,0.64,0.6272,0.36,0.36,0.00000,0,0.00000,0.00064,...,0.000000,0,0,0.00000,0,0.00000,0,0,0,0.000640


In [6]:
Feature_df

,Pb-2,Ti-4,Bi-3,Sc-3,Ni-2,Zr-4,Mn-2,Mn-3,Zn-2,Yb-3,Ga-3,In-3,Nb-5,Sn-2,Cd-2,Sb-3,Sb-5,Mg-2,Li-1,W-6
position,A,B,A,B,B,B,B,B,B,B,B,B,B,B,B,B,B,B,A,B
RSC6,119,60.5,103,74.5,69,72,83,64.5,74,86.8,62,80,64,118,95,76,60,72,76,60
RSC12,149,0,138.01,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,115.93,0
RC,154,132,152,144,115,145,117,117,125,170,125,150,134,140,141,141,141,136,123,130
RP,2.09,2.58,1.997,2.75,2.18,2.825,2.22,2.22,1.88,3.59,1.695,2.05,2.76,1.88,2.215,1.765,1.765,2.03,1.61,2.735
Rdce,1.43,0.91,1.38,1.1,1.13,0.98,0.79,0.79,1.25,1,1.38,1.49,0.95,1.41,1.39,1.32,1.32,1.09,1.75,0.92
Rdve,2.32,1.81,2.25,2.21,2.9,1.96,1.26,1.26,2.21,1.99,2,2.3,1.75,2.3,2.4,2.15,2.15,2.3,3.06,1.5
Ven,4,4,5,3,10,4,7,7,12,16,3,3,5,4,12,5,5,2,1,6
Ne/IR,0.536913,0.297521,0.579668,0.241611,0.376812,0.5,0.277108,0.341085,0.378378,0.771889,0.451613,0.575,0.5625,0.40678,0.484211,0.631579,0.766667,0.138889,0.017252,1.133333
CNE_C/IR,0.082483,0.079669,0.09666,0.062148,0.082754,0.089583,0.063012,0.081085,0.080676,0.098963,0.100323,0.105875,0.104688,0.077119,0.086211,0.131447,0.1665,0.045972,0.011041,0.164167


In [7]:
def calculate_weighted_average(properties_df, probabilities_df, position):
    """
    计算各种性质的加权平均数。
    """
    # 过滤出所需位置（A或B）的元素
    elements = properties_df.columns[properties_df.loc['position'] == position]
    
    # 筛选出这些元素的概率
    probabilities = probabilities_df[elements]
    
    # 计算加权平均数
    weighted_averages = {}
    for property_name in properties_df.index:
        if property_name == 'position':
            continue  # 跳过位置行
        weighted_averages[property_name + '_' +position] = (
            properties_df.loc[property_name, elements] * probabilities
        ).sum(axis=1) / probabilities.sum(axis=1)
    
    # 将结果转换为DataFrame
    weighted_averages_df = pd.DataFrame(weighted_averages)
    
    return weighted_averages_df

def calc_tolerance_factor(r_A, r_B, r_X=140):
    """
    计算容忍因子。
    """
    t = (r_A + r_X) / (1.414 * (r_B + r_X))
    return t

In [8]:
weighted_averages_df_A = calculate_weighted_average(Feature_df,Data_df,'A').drop(columns='RSC6_A')
weighted_averages_df_B = calculate_weighted_average(Feature_df,Data_df,'B').drop(columns='RSC12_B')

tolerance_factor = pd.DataFrame(calc_tolerance_factor(weighted_averages_df_A['RSC12_A'], weighted_averages_df_B['RSC6_B']),columns=['t'])

Feature_combined_df = pd.concat([Data_df.iloc[:,:2],weighted_averages_df_A,weighted_averages_df_B,tolerance_factor],axis=1)

In [9]:
Feature_combined_df #examination

,d33,Tc,RSC12_A,RC_A,RP_A,Rdce_A,Rdve_A,Ven_A,Ne/IR_A,CNE_C/IR_A,...,EN_MB_B,EN-P_B,EA_B,Period_B,RVdw_B,I1_B,I2_B,I3_B,OF_B,t
0,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.729594,1.427776,11.472514,4.000644,250.344062,648.285066,1283.114258,2557.539105,0.46837,0.980615
1,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.72878,1.427549,11.566106,4.001295,250.368404,648.290191,1283.229848,2557.113305,0.468599,0.980461
2,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.727955,1.42732,11.660795,4.001954,250.393032,648.295376,1283.346793,2556.682514,0.468832,0.980306
3,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.727121,1.427088,11.7566,4.00262,250.41795,648.300622,1283.465116,2556.246643,0.469067,0.98015
4,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.729685,1.427647,11.422304,4.000644,250.345671,648.278854,1283.035726,2557.492436,0.468402,0.980593
5,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.728961,1.42729,11.465102,4.001295,250.371641,648.277695,1283.071868,2557.019424,0.468664,0.980418
6,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.728229,1.426929,11.508401,4.001954,250.397916,648.276522,1283.108434,2556.540866,0.468929,0.980241
7,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.727488,1.426564,11.552211,4.00262,250.4245,648.275336,1283.145431,2556.056666,0.469198,0.980062
8,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.729491,1.427679,11.425201,4.000644,250.34374,648.339685,1283.107821,2557.680721,0.468381,0.980607
9,0,0,145.0436,153.28,2.05652,1.412,2.2948,4.36,0.552305,0.087587,...,1.728572,1.427355,11.470929,4.001295,250.367757,648.400065,1283.216899,2557.398187,0.468623,0.980446


In [10]:
file_path = "Feature_selected.xlsx"
Feature_group_df = pd.read_excel(file_path, header=0)

In [11]:
features = Feature_group_df["Feature"].tolist()

In [12]:
selected_Feature = pd.concat([Feature_combined_df.iloc[:,:2],Feature_combined_df[Feature_combined_df.columns.intersection(features)]],axis=1)

In [26]:
selected_Feature.to_excel("pre_data.xlsx")

In [16]:
import joblib
from sklearn.preprocessing import StandardScaler

best_model_d = joblib.load('best_xgb_model.pkl')
best_model_t = joblib.load('best_BR_model.pkl')
scaler = joblib.load('scaler_X.pkl')

In [19]:
selected_Feature.iloc[:,2:]

,RSC12_A,RC_A,Ne/IR_A,EA_A,RVdw_A,RSC6_B,RC_B,RP_B,Rdce_B,Rdve_B,...,CNE_S_B,IP_B,Electrons_B,EN-A_B,EN_MB_B,EN-P_B,EA_B,I1_B,I3_B,t
0,145.0436,153.28,0.552305,55.332,263.36,65.571773,136.338912,2.641476,0.97887,1.955085,...,3.096363,0.221634,18.018668,1.276714,1.729594,1.427776,11.472514,648.285066,2557.539105,0.980615
1,145.0436,153.28,0.552305,55.332,263.36,65.603917,136.358045,2.641755,0.979346,1.956183,...,3.09673,0.221911,18.037553,1.276627,1.72878,1.427549,11.566106,648.290191,2557.113305,0.980461
2,145.0436,153.28,0.552305,55.332,263.36,65.636438,136.377401,2.642037,0.979827,1.957294,...,3.097102,0.222192,18.056659,1.276539,1.727955,1.42732,11.660795,648.295376,2556.682514,0.980306
3,145.0436,153.28,0.552305,55.332,263.36,65.669342,136.396987,2.642322,0.980314,1.958418,...,3.097478,0.222476,18.075991,1.276449,1.727121,1.427088,11.7566,648.300622,2556.246643,0.98015
4,145.0436,153.28,0.552305,55.332,263.36,65.576279,136.339556,2.641489,0.978761,1.954557,...,3.096218,0.221679,18.017702,1.276666,1.729685,1.427647,11.422304,648.278854,2557.492436,0.980593
5,145.0436,153.28,0.552305,55.332,263.36,65.612982,136.35934,2.641781,0.979126,1.955121,...,3.096439,0.222003,18.03561,1.27653,1.728961,1.42729,11.465102,648.277695,2557.019424,0.980418
6,145.0436,153.28,0.552305,55.332,263.36,65.650114,136.379355,2.642076,0.979495,1.955692,...,3.096662,0.22233,18.053728,1.276392,1.728229,1.426929,11.508401,648.276522,2556.540866,0.980241
7,145.0436,153.28,0.552305,55.332,263.36,65.687684,136.399607,2.642375,0.979869,1.956269,...,3.096888,0.22266,18.07206,1.276253,1.727488,1.426564,11.552211,648.275336,2556.056666,0.980062
8,145.0436,153.28,0.552305,55.332,263.36,65.573383,136.342131,2.641379,0.978909,1.954863,...,3.09646,0.221603,18.019311,1.276685,1.729491,1.427679,11.425201,648.339685,2557.680721,0.980607
9,145.0436,153.28,0.552305,55.332,263.36,65.607154,136.364519,2.64156,0.979424,1.955736,...,3.096925,0.221848,18.038848,1.276568,1.728572,1.427355,11.470929,648.400065,2557.398187,0.980446


In [20]:
X_new_scaled = scaler.transform(selected_Feature.iloc[:,2:])

In [59]:
Predict_d = best_model_d.predict(X_new_scaled)
Predict_t = best_model_t.predict(X_new_scaled)

c:\Users\zzp\miniconda3\envs\Ceramic\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but BayesianRidge was fitted with feature names
  warnings.warn(


In [22]:
Predict_d

array([409.51978, 343.8099 , 292.9977 , 257.74942, 379.94177, 289.3931 ,
       289.60052, 256.05896, 414.59924, 359.25415, 321.47498, 311.1123 ,
       394.0511 , 382.90204, 284.40952, 272.68274, 413.05652, 310.0937 ,
       317.30133, 268.3361 , 434.73166, 430.11172, 452.3691 , 407.15714,
       374.72708, 295.23233, 347.81705, 422.19742, 313.53662, 335.339  ,
       357.09094, 376.03473], dtype=float32)

In [23]:
Predict_t

array([433.42437883, 431.62571125, 429.80607428, 427.96509901,
       433.94841076, 432.67991662, 431.39670317, 430.09851267,
       433.74386277, 432.26841422, 430.77579723, 429.26571033,
       434.31612186, 433.41959688, 432.51268507, 431.59520482,
       433.57092747, 431.92058378, 430.25107831, 428.56207528,
       434.92440238, 434.64772461, 434.37240193, 434.09843217,
       435.9193232 , 436.63621207, 437.35310407, 438.0699992 ,
       433.1069586 , 431.01222124, 428.91822448, 426.82496744])

In [75]:
new_df = pd.read_excel('newdata.xlsx', sheet_name=0, header=0) 

In [76]:
new_df

,d33,Tc,RSC12_A,RC_A,Ne/IR_A,EA_A,RVdw_A,RSC6_B,RC_B,RP_B,...,CNE_S_B,IP_B,Electrons_B,EN-A_B,EN_MB_B,EN-P_B,EA_B,I1_B,I3_B,t
0,434,426,145.0436,153.28,0.552305,55.332,263.36,65.571773,136.338912,2.641476,...,3.096363,0.221634,18.018668,1.276714,1.729594,1.427776,11.472514,648.285066,2557.539105,0.980615
1,430,424,145.0436,153.28,0.552305,55.332,263.36,65.573383,136.342131,2.641379,...,3.096460,0.221603,18.019311,1.276685,1.729491,1.427679,11.425201,648.339685,2557.680721,0.980607
2,445,0,145.0436,153.28,0.552305,55.332,263.36,65.607154,136.364519,2.641560,...,3.096925,0.221848,18.038848,1.276568,1.728572,1.427355,11.470929,648.400065,2557.398187,0.980446
3,465,0,145.0436,153.28,0.552305,55.332,263.36,65.641322,136.387170,2.641744,...,3.097395,0.222096,18.058613,1.276451,1.727642,1.427027,11.517193,648.461153,2557.112341,0.980283
4,451,0,145.0436,153.28,0.552305,55.332,263.36,65.675893,136.410088,2.641929,...,3.097871,0.222347,18.078611,1.276331,1.726702,1.426695,11.564003,648.522961,2556.823125,0.980118
5,500,0,145.0436,153.28,0.552305,55.332,263.36,65.572739,136.345671,2.641427,...,3.095977,0.221540,18.013518,1.276547,1.729450,1.427551,11.415546,648.285388,2558.935951,0.980610


In [70]:
new_df.iloc[:1,2:]

,RSC12_A,RC_A,Ne/IR_A,EA_A,RVdw_A,RSC6_B,RC_B,RP_B,Rdce_B,Rdve_B,...,CNE_S_B,IP_B,Electrons_B,EN-A_B,EN_MB_B,EN-P_B,EA_B,I1_B,I3_B,t
0,145.0436,153.28,0.552305,55.332,263.36,65.571773,136.338912,2.641476,0.97887,1.955085,...,3.096363,0.221634,18.018668,1.276714,1.729594,1.427776,11.472514,648.285066,2557.539105,0.980615


In [71]:
Predict_d = best_model_d.predict(X_new_scaled)
Predict_t = best_model_t.predict(X_new_scaled)

c:\Users\zzp\miniconda3\envs\Ceramic\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but BayesianRidge was fitted with feature names
  warnings.warn(


In [ ]:

best_model_d.fit(
    new_df.iloc[:2,2:], 
    new_df.iloc[:2,:1], 
    xgb_model=best_model_d.get_booster()   # 指定继续训练
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=np.float64(0.6588293923716152), device=None,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, feature_types=None, feature_weights=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None,
             learning_rate=np.float64(0.1953175250322989), max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=607, n_jobs=-1,
             num_parallel_tree=None, ...)

In [78]:
 new_df.iloc[:2,1:2]

,Tc
0,426
1,424


In [ ]:

best_model_t.fit(
    new_df.iloc[:2,2:], 
    new_df.iloc[:2,1:2], 
    xgb_model=best_model_t.get_booster()   # 指定继续训练
)

AttributeError: 'BayesianRidge' object has no attribute 'get_booster'

In [73]:
Predict_d = best_model_d.predict(X_new_scaled)

In [74]:
Predict_d 

array([510.0294 , 444.31958, 393.50732, 358.23233, 480.45135, 389.9027 ,
       390.11017, 356.5419 , 515.10895, 459.76373, 421.98468, 411.62213,
       494.56067, 483.41162, 384.91925, 373.16574, 513.5662 , 410.60345,
       417.81104, 368.81906, 535.2414 , 530.62146, 552.8788 , 507.6668 ,
       475.2367 , 395.74194, 448.32666, 522.70715, 414.0465 , 435.84863,
       457.6006 , 476.5443 ], dtype=float32)